# SC-10-Account-Abstraction - ERC-4337

[<< DAO Governance](SC-9-DAO-Governance.ipynb) | [LLM Assisted >>](SC-11-LLM-Assisted.ipynb)

---

## Objectifs d'apprentissage

1. Comprendre l'**account abstraction** (ERC-4337)
2. Creer des **UserOperations**
3. Implementer un **Smart Account** simple
4. Comprendre les **Paymasters**

### Prerequis

- [SC-1](../01-Solidity-Foundation/SC-3-Solidity-Basics.ipynb) a [SC-7](SC-9-DAO-Governance.ipynb) completes
- Comprendre le modele de compte Ethereum (EOA vs contrats)
- Notions sur ERC-4337 (optionnel, couvert dans le notebook)

### Duree estimee : 50 minutes

---

## 0. Connexion a la blockchain locale

Tous les contrats de ce notebook sont **compiles et deployes reellement** sur anvil.
Lancez `anvil` dans un terminal avant d'executer les cellules.


---

## 1. Concepts ERC-4337

ERC-4337 permet de separer la logique des comptes des wallets externes.

| Composant | Description |
|-----------|-------------|
| **UserOperation** | Transaction utilisateur (comme transfer, swap) |
| **EntryPoint** | Contrat singleton qui traite les UserOps |
| **Smart Account** | Wallet intelligent (logique metier en contrat) |
| **Paymaster** | Sponsor de frais de gas |
| **Bundler** | Empaquete les UserOps en batch |

In [1]:
# %% Cellule - Configuration Web3 avec import guards
# Guard: verification des dependances optionnelles

try:
    from web3 import Web3
    WEB3_AVAILABLE = True
    print('web3.py disponible.')
except ImportError:
    WEB3_AVAILABLE = False
    print('web3.py non disponible. Les cellules utilisant Web3 seront skippees.')

try:
    import solcx
    SOLCX_AVAILABLE = True
    print('py-solc-x disponible.')
except ImportError:
    SOLCX_AVAILABLE = False
    print('py-solc-x non disponible. Compilation Solc skippee.')

import sys, os
sys.path.insert(0, os.path.abspath(".."))
try:
    from forge_helper import forge_compile, forge_compile_and_deploy
    FORGE_HELPER_AVAILABLE = True
    print('forge_helper disponible.')
except ImportError:
    FORGE_HELPER_AVAILABLE = False
    print('forge_helper non disponible. Compilation Foundry skippee.')

deployer = None

if WEB3_AVAILABLE:
    # Connexion au provider (anvil local ou autre RPC)
    w3 = Web3(Web3.HTTPProvider('http://127.0.0.1:8545'))
    connected = w3.is_connected()
    print(f'Connexion Web3: {"OK" if connected else "ECHOUEE"}')

    # Compte deployeur = premier compte finance par anvil
    if connected:
        deployer = w3.eth.accounts[0]
        print(f'Deployeur: {deployer}')

    if not SOLCX_AVAILABLE:
        print('Note: py-solc-x non installe. Utilisez forge_helper ou installez avec pip install py-solc-x')

    # Fonction utilitaire de compilation (si solcx disponible).
    # Version solc alignee sur le pragma des contrats (^0.8.28) et la chaine Foundry.
    if SOLCX_AVAILABLE:
        try:
            solcx.install_solc('0.8.28')
            solcx.set_solc_version('0.8.28')
            print(f'solcx version: {solcx.get_solc_version()}')
        except Exception as e:
            print(f'Erreur configuration solcx: {e}')
            SOLCX_AVAILABLE = False

    def compile_and_deploy(w3_instance, contract_source, constructor_arg=None, deploy_from=None):
        # Compile et deploie un contrat Solidity via solcx.
        # constructor_arg : argument unique passe au constructeur (None = aucun).
        # deploy_from     : compte emetteur de la transaction de deploiement.
        if not SOLCX_AVAILABLE:
            print('solcx non disponible pour compilation.')
            return None, None
        try:
            compiled = solcx.compile_source(
                contract_source,
                output_values=['abi', 'bin'],
                solc_version='0.8.28'
            )
            contract_id, contract_interface = compiled.popitem()
            contract_name = contract_id.split(':')[-1]
            abi = contract_interface['abi']
            bytecode = contract_interface['bin']
            Contract = w3_instance.eth.contract(abi=abi, bytecode=bytecode)
            sender = deploy_from if deploy_from is not None else w3_instance.eth.accounts[0]
            # constructor_arg peut etre un scalaire (1 arg) ou une liste/tuple (plusieurs args)
            if constructor_arg is None:
                ctor_args = []
            elif isinstance(constructor_arg, (list, tuple)):
                ctor_args = list(constructor_arg)
            else:
                ctor_args = [constructor_arg]
            tx_hash = Contract.constructor(*ctor_args).transact({'from': sender})
            tx_receipt = w3_instance.eth.wait_for_transaction_receipt(tx_hash)
            contract_address = tx_receipt['contractAddress']
            deployed_contract = w3_instance.eth.contract(address=contract_address, abi=abi)
            print(f"Deploye: {contract_name} a {contract_address}")
            return deployed_contract, tx_receipt
        except Exception as e:
            print(f'Erreur compilation/deployment: {e}')
            return None, None
else:
    print('Web3 non disponible. Cellules dependantes desactivees.')
# UserOperation structure
USER_OP_STRUCT = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

struct UserOperation {
    address sender;
    uint256 nonce;
    bytes initCode;
    bytes callData;
    uint256 callGasLimit;
    uint256 verificationGasLimit;
    uint256 preVerificationGas;
    uint256 maxFeePerGas;
    uint256 maxPriorityFeePerGas;
    bytes paymasterAndData;
    bytes signature;
}
'''

print("Structure UserOperation (ERC-4337):")
print(USER_OP_STRUCT)
print("Chaque champ controle un aspect de la transaction :")
print("  sender          = adresse du Smart Account")
print("  nonce           = protection contre le replay")
print("  callData        = la transaction a executer")
print("  signature       = preuve d'autorisation")
print("  paymasterAndData = donnees du paymaster (si gas sponsorise)")

web3.py disponible.
py-solc-x disponible.
forge_helper disponible.
Connexion Web3: OK
Deployeur: 0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266
solcx version: 0.8.28
Structure UserOperation (ERC-4337):

// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

struct UserOperation {
    address sender;
    uint256 nonce;
    bytes initCode;
    bytes callData;
    uint256 callGasLimit;
    uint256 verificationGasLimit;
    uint256 preVerificationGas;
    uint256 maxFeePerGas;
    uint256 maxPriorityFeePerGas;
    bytes paymasterAndData;
    bytes signature;
}

Chaque champ controle un aspect de la transaction :
  sender          = adresse du Smart Account
  nonce           = protection contre le replay
  callData        = la transaction a executer
  signature       = preuve d'autorisation
  paymasterAndData = donnees du paymaster (si gas sponsorise)


### Interprétation : Structure UserOperation ERC-4337

**Sortie obtenue** : Affichage de la structure `UserOperation` et explication des champs.

| Champ | Type | Description |
|-------|------|-------------|
| `sender` | address | Adresse du Smart Account (créateur de l'opération) |
| `nonce` | uint256 | Compteur d'opérations (anti-replay) |
| `initCode` | bytes | Bytecode de création du compte (si compte inexistant) |
| `callData` | bytes | Données de l'appel à exécuter |
| `callGasLimit` | uint256 | Gas maximum pour l'exécution de l'appel |
| `verificationGasLimit` | uint256 | Gas pour la validation de signature |
| `preVerificationGas` | uint256 | Gas pour les frais fixes (remboursement) |
| `maxFeePerGas` | uint256 | Prix max du gas (EIP-1559) |
| `maxPriorityFeePerGas` | uint256 | Pourboire au validateur (EIP-1559) |
| `paymasterAndData` | bytes | Adresse du paymaster + données de sponsoring |
| `signature` | bytes | Signature de l'utilisateur (preuve d'autorisation) |

**Points clés** :
1. ERC-4337 introduit une nouvelle abstraction de compte qui remplace les EOA (Externally Owned Accounts)
2. La `UserOperation` est une pseudo-transaction qui sera traitée par l'EntryPoint
3. Le `paymasterAndData` permet de sponsoriser les frais de gas (gasless transactions)

> **Note technique** : Contrairement à une transaction Ethereum classique, une UserOperation ne contient pas de `value` ni de `to` directs. Ces informations sont encodées dans le `callData`.


---

## 2. Smart Account Simple

Un smart account est un contrat qui peut executer des transactions.

In [2]:
# Simple Smart Account (ERC-4337)
# Ce contrat herite de BaseAccount (account-abstraction SDK).
# Il necessite un EntryPoint deploye pour fonctionner en production.
# Ici on compile avec Foundry pour verifier la validite du code.

SIMPLE_ACCOUNT = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

import "@account-abstraction/contracts/core/BaseAccount.sol";
import "@account-abstraction/contracts/core/Helpers.sol";
import "@openzeppelin/contracts/utils/cryptography/ECDSA.sol";
import "@openzeppelin/contracts/utils/cryptography/MessageHashUtils.sol";

contract SimpleAccount is BaseAccount {
    using ECDSA for bytes32;

    address public owner;
    IEntryPoint private immutable _entryPoint;

    event SimpleAccountInitialized(IEntryPoint indexed entryPoint, address indexed owner);

    constructor(address anOwner, IEntryPoint anEntryPoint) {
        owner = anOwner;
        _entryPoint = anEntryPoint;
    }

    function entryPoint() public view override returns (IEntryPoint) {
        return _entryPoint;
    }

    // Verification de signature
    function _validateSignature(
        PackedUserOperation calldata userOp,
        bytes32 userOpHash
    ) internal virtual override returns (uint256 validationData) {
        bytes32 hash = MessageHashUtils.toEthSignedMessageHash(userOpHash);
        if (owner != ECDSA.recover(hash, userOp.signature)) {
            return SIG_VALIDATION_FAILED;
        }
        return SIG_VALIDATION_SUCCESS;
    }

    // Autoriser owner en plus de l EntryPoint
    function _requireForExecute() internal view override {
        require(
            msg.sender == address(_entryPoint) || msg.sender == owner,
            "only entry point or owner"
        );
    }

    receive() external payable {}
}
'''

# Compilation avec Foundry (resout les imports @account-abstraction et @openzeppelin)
abi, bytecode = forge_compile(SIMPLE_ACCOUNT, "SimpleAccount")
print(f"Compilation reussie ! ABI: {len(abi)} fonctions/events")
print(f"Bytecode: {len(bytecode)} caracteres")
print()

# Afficher les fonctions publiques du contrat compile
print("Fonctions et events du SimpleAccount:")
for item in abi:
    if item.get('type') == 'function':
        inputs = ', '.join(f"{i['type']} {i['name']}" for i in item.get('inputs', []))
        print(f"  function {item['name']}({inputs})")
    elif item.get('type') == 'event':
        print(f"  event {item['name']}")

# Note: le deploiement necessite un EntryPoint reel.
print()
print("Note: deploiement impossible sans EntryPoint reel.")
print("Voir la version standalone ci-dessous pour la demo deployable.")


Compilation reussie ! ABI: 14 fonctions/events
Bytecode: 9982 caracteres

Fonctions et events du SimpleAccount:
  function entryPoint()
  function execute(address target, uint256 value, bytes data)
  function executeBatch(tuple[] calls)
  function getNonce()
  function owner()
  function validateUserOp(tuple userOp, bytes32 userOpHash, uint256 missingAccountFunds)
  event SimpleAccountInitialized

Note: deploiement impossible sans EntryPoint reel.
Voir la version standalone ci-dessous pour la demo deployable.


### Interprétation : Compilation du SimpleAccount avec Foundry

**Sortie obtenue** : Compilation réussie avec Foundry, affichage de l'ABI et des fonctions.

| Métrique | Valeur | Signification |
|----------|--------|---------------|
| ABI length | 14 fonctions/events | Interface complète du contrat |
| Bytecode length | 9982 caractères | ~5 ko de bytecode (contrat complexe) |
| Compilation status | SUCCESS | Le code est valide Solidity 0.8.28 |

**Analyse des fonctions principales** :

| Fonction | Description | Héritée de |
|----------|-------------|------------|
| `entryPoint()` | Retourne l'adresse de l'EntryPoint | BaseAccount |
| `execute()` | Exécute une transaction | BaseAccount |
| `executeBatch()` | Exécute plusieurs transactions en lot | BaseAccount |
| `validateUserOp()` | Valide la signature de l'utilisateur | BaseAccount |
| `owner()` | Retourne le propriétaire du compte | SimpleAccount |

**Points clés** :
1. `SimpleAccount` hérite de `BaseAccount` qui gère la logique ERC-4337
2. La fonction `_validateSignature()` implémente la vérification ECDSA
3. L'ABI contient les événements pour le suivi des opérations

> **Note technique** : La compilation avec Foundry (`forge_compile`) résout automatiquement les imports `@account-abstraction` et `@openzeppelin` depuis le répertoire `lib/` du projet Foundry.


### Version standalone deployable

Le contrat ci-dessus depend d'un EntryPoint deploye. Voici une version
autonome qui illustre les memes concepts (owner, execution, validation)
sans dependances externes.

In [3]:
# Version standalone du Smart Account (sans dependances ERC-4337)
# Deploiement reel sur anvil

STANDALONE_ACCOUNT = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

/// @title StandaloneSmartAccount
/// @notice Version simplifiee d un smart account ERC-4337 pour demonstration.
/// Illustre: owner, execution de transactions, batch, nonce.
contract StandaloneSmartAccount {
    address public owner;
    uint256 public nonce;

    event Executed(address indexed dest, uint256 value, bytes data);
    event OwnerChanged(address indexed oldOwner, address indexed newOwner);

    modifier onlyOwner() {
        require(msg.sender == owner, "not owner");
        _;
    }

    constructor(address _owner) {
        owner = _owner;
    }

    // Executer une transaction
    function execute(address dest, uint256 value, bytes calldata data) external onlyOwner {
        nonce++;
        (bool success, ) = dest.call{value: value}(data);
        require(success, "execution failed");
        emit Executed(dest, value, data);
    }

    // Executer un batch
    function executeBatch(
        address[] calldata dests,
        uint256[] calldata values,
        bytes[] calldata datas
    ) external onlyOwner {
        require(dests.length == values.length && dests.length == datas.length, "length mismatch");
        for (uint256 i = 0; i < dests.length; i++) {
            nonce++;
            (bool success, ) = dests[i].call{value: values[i]}(datas[i]);
            require(success, "batch call failed");
            emit Executed(dests[i], values[i], datas[i]);
        }
    }

    // Changer le owner (social recovery, etc.)
    function changeOwner(address newOwner) external onlyOwner {
        require(newOwner != address(0), "zero address");
        emit OwnerChanged(owner, newOwner);
        owner = newOwner;
    }

    receive() external payable {}
}
'''

# Deployer sur anvil
account, receipt = compile_and_deploy(w3, STANDALONE_ACCOUNT, deployer, deployer)

# Verifier le owner
print(f"Owner: {account.functions.owner().call()}")
print(f"Nonce: {account.functions.nonce().call()}")

# Envoyer de l ETH au smart account
tx = w3.eth.send_transaction({
    'from': deployer,
    'to': account.address,
    'value': w3.to_wei(1, 'ether')
})
w3.eth.wait_for_transaction_receipt(tx)
balance = w3.eth.get_balance(account.address)
print(f"Balance du smart account: {w3.from_wei(balance, 'ether')} ETH")


Deploye: StandaloneSmartAccount a 0x8A791620dd6260079BF849Dc5567aDC3F2FdC318
Owner: 0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266
Nonce: 0
Balance du smart account: 1 ETH


### Interprétation : Déploiement et test du StandaloneSmartAccount

**Sortie obtenue** : Contrat déployé avec succès sur anvil, état initial affiché.

| Métrique | Valeur | Signification |
|----------|--------|---------------|
| Adresse | `0x8A791620dd6260079BF849Dc5567aDC3F2FdC318` | Adresse du contrat sur anvil |
| Owner | `0xf39Fd6...92266` | Compte propriétaire (deployer) |
| Nonce | 0 | Compteur d'opérations initialisé à 0 |
| Balance | 1 ETH | Solde du smart account après envoi |

**Flux d'exécution** :
1. **Construction** : Le contrat est créé avec le `deployer` comme owner
2. **Funding** : Envoi de 1 ETH au smart account pour les futures transactions
3. **Vérification** : Lecture de l'état (`owner()`, `nonce()`, balance)

**Points clés** :
1. Le `nonce` est incrémenté à chaque appel de `execute()` ou `executeBatch()`
2. Le modificateur `onlyOwner` restreint l'exécution au propriétaire uniquement
3. Le smart account peut recevoir et stocker de l'ETH (fonction `receive()`)

> **Note technique** : Cette version standalone illustre les concepts ERC-4337 sans dépendances. En production, utilisez l'infrastructure complète avec EntryPoint et Bundler.


### Exercice 2 : Smart Account avec liste blanche

Ajoutez un mecanisme de liste blanche au smart account : seules les adresses autorisees peuvent recevoir des fonds via `execute()`.

**Objectif** : Comprendre comment un smart account peut restreindre ses transactions a un ensemble de destinataires de confiance.

**Indice** : Utilisez un `mapping(address => bool)` pour la liste blanche. Ajoutez des fonctions `addToWhitelist(address)` et `removeFromWhitelist(address)` reservees au owner. Dans `execute()`, ajoutez un `require(isWhitelisted[dest])` avant l'appel externe.

In [13]:
# Exercice 2 : Smart Account avec liste blanche
# TODO etudiant : implementer un smart account avec une whitelist de destinataires autorises
# Indice : ajoutez un mapping(address => bool) pour la whitelist
# Etape 1 : implementer addToWhitelist(address) et removeFromWhitelist(address) - onlyOwner
# Etape 2 : modifier execute() pour verifier que dest est dans la whitelist
# Etape 3 : implementer isWhitelisted(address) - view

EXERCISE_WHITELIST_ACCOUNT = """
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract WhitelistAccount {
    address public owner;
    uint256 public nonce;

    // TODO etudiant : mapping de la liste blanche
    // Indice : mapping(address => bool) private _whitelist;

    event Executed(address indexed dest, uint256 value, bytes data);
    event WhitelistAdded(address indexed account);
    event WhitelistRemoved(address indexed account);

    modifier onlyOwner() {
        require(msg.sender == owner, \"not owner\");
        _;
    }

    constructor() {
        owner = msg.sender;
    }

    function addToWhitelist(address account) external onlyOwner {
        // TODO etudiant : ajouter account a la liste blanche
        // Indice : _whitelist[account] = true
        // TODO etudiant : emettre WhitelistAdded
    }

    function removeFromWhitelist(address account) external onlyOwner {
        // TODO etudiant : retirer account de la liste blanche
        // Indice : _whitelist[account] = false
        // TODO etudiant : emettre WhitelistRemoved
    }

    function isWhitelisted(address account) public view returns (bool) {
        // TODO etudiant : retourner le statut de la liste blanche
        return false;
    }

    function execute(address dest, uint256 value, bytes calldata data) external onlyOwner {
        // TODO etudiant : verifier que dest est dans la liste blanche
        // Indice : require(_whitelist[dest], \"dest not whitelisted\")
        // TODO etudiant : incrementer nonce, executer l appel, emettre Executed
    }

    receive() external payable {}
}
"""

print("Exercice a completer : WhitelistAccount avec liste blanche de destinataires")

Exercice a completer : WhitelistAccount avec liste blanche de destinataires


---

## 3. Paymaster

Un paymaster peut payer les frais de gas pour les utilisateurs.

In [4]:
# Verifying Paymaster (ERC-4337)
# Un paymaster peut sponsoriser les frais de gas.
# Ce contrat herite de BasePaymaster (account-abstraction SDK)
# et necessite un EntryPoint deploye.

PAYMASTER = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

import "@account-abstraction/contracts/core/BasePaymaster.sol";
import "@account-abstraction/contracts/core/Helpers.sol";
import "@openzeppelin/contracts/utils/cryptography/ECDSA.sol";
import "@openzeppelin/contracts/utils/cryptography/MessageHashUtils.sol";

contract VerifyingPaymaster is BasePaymaster {
    using ECDSA for bytes32;

    address public verifyingSigner;

    constructor(IEntryPoint _entryPoint, address _verifyingSigner)
        BasePaymaster(_entryPoint, msg.sender)
    {
        verifyingSigner = _verifyingSigner;
    }

    // Validation du paymaster
    function _validatePaymasterUserOp(
        PackedUserOperation calldata userOp,
        bytes32 userOpHash,
        uint256 maxCost
    ) internal override returns (bytes memory context, uint256 validationData) {
        (uint48 validUntil, uint48 validAfter, bytes calldata signature) =
            parsePaymasterAndData(userOp.paymasterAndData);

        bytes32 hash = keccak256(abi.encode(userOpHash, validUntil, validAfter));
        bytes32 ethSignedHash = MessageHashUtils.toEthSignedMessageHash(hash);

        if (verifyingSigner != ECDSA.recover(ethSignedHash, signature)) {
            return ("", _packValidationData(true, validUntil, validAfter));
        }

        return ("", _packValidationData(false, validUntil, validAfter));
    }

    function parsePaymasterAndData(bytes calldata paymasterAndData)
        internal pure
        returns (uint48 validUntil, uint48 validAfter, bytes calldata signature)
    {
        require(paymasterAndData.length >= 52, "invalid paymasterAndData");
        validUntil = uint48(bytes6(paymasterAndData[20:26]));
        validAfter = uint48(bytes6(paymasterAndData[26:32]));
        signature = paymasterAndData[32:];
    }
}
'''

# Compilation avec Foundry
abi_pm, bytecode_pm = forge_compile(PAYMASTER, "VerifyingPaymaster")
print(f"Compilation reussie ! ABI: {len(abi_pm)} fonctions/events")
print(f"Bytecode: {len(bytecode_pm)} caracteres")
print()

# Afficher les fonctions publiques
print("Fonctions et events du VerifyingPaymaster:")
for item in abi_pm:
    if item.get('type') == 'function':
        inputs = ', '.join(f"{i['type']} {i['name']}" for i in item.get('inputs', []))
        print(f"  function {item['name']}({inputs})")
    elif item.get('type') == 'event':
        print(f"  event {item['name']}")

print()
print("Note: deploiement impossible sans EntryPoint reel (interface ERC-165 requise).")
print("En production, le Paymaster est deploye apres l'EntryPoint.")


Compilation reussie ! ABI: 26 fonctions/events
Bytecode: 15108 caracteres

Fonctions et events du VerifyingPaymaster:
  function acceptOwnership()
  function addStake(uint32 unstakeDelaySec)
  function deposit()
  function entryPoint()
  function getDeposit()
  function owner()
  function pendingOwner()
  function postOp(uint8 mode, bytes context, uint256 actualGasCost, uint256 actualUserOpFeePerGas)
  function renounceOwnership()
  function transferOwnership(address newOwner)
  function unlockStake()
  function validatePaymasterUserOp(tuple userOp, bytes32 userOpHash, uint256 maxCost)
  function verifyingSigner()
  function withdrawStake(address withdrawAddress)
  function withdrawTo(address withdrawAddress, uint256 amount)
  event OwnershipTransferStarted
  event OwnershipTransferred

Note: deploiement impossible sans EntryPoint reel (interface ERC-165 requise).
En production, le Paymaster est deploye apres l'EntryPoint.


### Interprétation : Compilation du VerifyingPaymaster

**Sortie obtenue** : Compilation réussie, affichage de l'ABI complète du Paymaster.

| Métrique | Valeur | Signification |
|----------|--------|---------------|
| ABI length | 26 fonctions/events | Interface étendue (gestion de stake, dépôts, retraits) |
| Bytecode length | 15108 caractères | ~7.5 ko de bytecode |
| Compilation status | SUCCESS | Le code est valide |

**Analyse des fonctions clés du Paymaster** :

| Catégorie | Fonctions | Description |
|-----------|-----------|-------------|
| **Staking** | `addStake()`, `unlockStake()`, `withdrawStake()` | Gestion du stake de sécurité (anti-DOS) |
| **Dépôts** | `deposit()`, `getDeposit()`, `withdrawTo()` | Gestion des fonds de sponsoring |
| **Validation** | `validatePaymasterUserOp()` | Validation de la signature du paymaster |
| **Ownership** | `owner()`, `transferOwnership()`, `acceptOwnership()` | Gestion de la propriété du contrat |

**Points clés** :
1. Le `VerifyingPaymaster` hérite de `BasePaymaster` qui implémente la logique ERC-4337
2. La fonction `_validatePaymasterUserOp()` vérifie qu'un signateur autorisé a approuvé l'opération
3. Le stake est requis pour éviter les attaques DOS (le paymaster perd son stake s'il fraude)

> **Note technique** : En production, le paymaster doit être staké (généralement ~1 ETH) et déposé avec des fonds pour sponsoriser les gas des utilisateurs.


### Version standalone deployable

Version simplifiee d'un paymaster qui illustre le mecanisme de sponsoring
de gas sans dependre de l'infrastructure EntryPoint.

In [5]:
# Version standalone du Paymaster (sans dependances ERC-4337)

STANDALONE_PAYMASTER = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

/// @title StandalonePaymaster
/// @notice Demonstre le concept de sponsoring de gas.
/// Le signer autorise des operations en signant off-chain.
contract StandalonePaymaster {
    address public owner;
    address public verifyingSigner;
    mapping(address => uint256) public deposits;

    event Deposited(address indexed account, uint256 amount);
    event Withdrawn(address indexed account, uint256 amount);
    event GasSponsored(address indexed account, uint256 amount);

    constructor(address _signer) {
        owner = msg.sender;
        verifyingSigner = _signer;
    }

    // Deposer des fonds pour sponsoriser le gas
    function deposit() external payable {
        deposits[msg.sender] += msg.value;
        emit Deposited(msg.sender, msg.value);
    }

    // Retirer des fonds
    function withdraw(uint256 amount) external {
        require(deposits[msg.sender] >= amount, "insufficient deposit");
        deposits[msg.sender] -= amount;
        payable(msg.sender).transfer(amount);
        emit Withdrawn(msg.sender, amount);
    }

    // Simuler le sponsoring de gas
    function sponsorGas(address account, uint256 gasAmount) external {
        require(msg.sender == owner, "only owner");
        require(deposits[owner] >= gasAmount, "insufficient funds");
        deposits[owner] -= gasAmount;
        emit GasSponsored(account, gasAmount);
    }

    function getDeposit(address account) external view returns (uint256) {
        return deposits[account];
    }
}
'''

# Deployer
paymaster, receipt = compile_and_deploy(w3, STANDALONE_PAYMASTER, deployer, deployer)

# Deposer des fonds
tx = paymaster.functions.deposit().transact({'from': deployer, 'value': w3.to_wei(5, 'ether')})
w3.eth.wait_for_transaction_receipt(tx)
deposit = paymaster.functions.getDeposit(deployer).call()
print(f"Deposit du paymaster: {w3.from_wei(deposit, 'ether')} ETH")

# Sponsoriser du gas pour un compte
user = w3.eth.accounts[1]
tx = paymaster.functions.sponsorGas(user, w3.to_wei(0.1, 'ether')).transact({'from': deployer})
w3.eth.wait_for_transaction_receipt(tx)
print(f"Gas sponsorise pour {user[:10]}... : 0.1 ETH")
deposit_after = paymaster.functions.getDeposit(deployer).call()
print(f"Deposit restant: {w3.from_wei(deposit_after, 'ether')} ETH")


Deploye: StandalonePaymaster a 0xB7f8BC63BbcaD18155201308C8f3540b07f84F5e
Deposit du paymaster: 5 ETH
Gas sponsorise pour 0x70997970... : 0.1 ETH
Deposit restant: 4.9 ETH


### Interprétation : Déploiement et test du StandalonePaymaster

**Sortie obtenue** : Paymaster déployé, dépôt de 5 ETH, sponsoring de 0.1 ETH pour un utilisateur.

| Métrique | Valeur | Signification |
|----------|--------|---------------|
| Adresse | `0xB7f8BC63BbcaD18155201308C8f3540b07f84F5e` | Adresse du paymaster sur anvil |
| Deposit initial | 5 ETH | Fonds déposés pour le sponsoring |
| Gas sponsorisé | 0.1 ETH | Montant alloué à l'utilisateur |
| Deposit restant | 4.9 ETH | Solde après sponsoring |

**Flux de sponsoring de gas** :
1. **Dépôt** : Le propriétaire dépose des fonds (`deposit()`) pour sponsoriser les gas
2. **Sponsoring** : Le propriétaire appelle `sponsorGas(user, amount)` pour allouer des fonds
3. **Utilisation** : L'utilisateur peut utiliser ces fonds pour payer ses gas fees
4. **Retrait** : Les fonds non utilisés peuvent être retirés (`withdraw()`)

**Points clés** :
1. Le paymaster agit comme un intermédiaire qui paie les gas au nom des utilisateurs
2. Cette architecture permet des "gasless transactions" (l'utilisateur ne paie pas directement)
3. En production, le paymaster vérifie la signature off-chain avant de sponsoriser

> **Note technique** : Dans un vrai déploiement ERC-4337, le paymaster rembourse le smart account après l'exécution de la UserOperation. Le montant remboursé est basé sur le gas réellement utilisé.


### Exercice 3 : Paymaster avec budget par utilisateur

Creez un `BudgetPaymaster` qui attribue un budget individuel a chaque utilisateur. Le sponsoring de gas ne doit pas depasser le budget alloue.

**Objectif** : Comprendre comment un paymaster peut limiter les frais sponsorises par utilisateur, evitant qu'un seul utilisateur ne consomme tout le budget.

**Indice** : Ajoutez deux mappings `budgets(address => uint256)` et `spent(address => uint256)`. La fonction `sponsorGas` doit verifier que `spent[user] + gasAmount <= budgets[user]` avant de debiter.

In [14]:
# Exercice 3 : Paymaster avec budget par utilisateur
# TODO etudiant : implementer un paymaster avec un budget individuel par utilisateur
# Indice : ajoutez un mapping(address => uint256) pour les budgets
# Etape 1 : implementer setUserBudget(address user, uint256 budget) - onlyOwner
# Etape 2 : implementer getUserBudget(address user) - view
# Etape 3 : modifier sponsorGas pour verifier spent[user] + gasAmount <= budgets[user]
# Etape 4 : ajouter un mapping(address => uint256) pour tracker les depenses par utilisateur

EXERCISE_BUDGET_PAYMASTER = """
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract BudgetPaymaster {
    address public owner;
    address public verifyingSigner;

    // TODO etudiant : mapping des budgets alloues par utilisateur
    // Indice : mapping(address => uint256) public budgets;

    // TODO etudiant : mapping des depenses par utilisateur
    // Indice : mapping(address => uint256) public spent;

    mapping(address => uint256) public deposits;

    event Deposited(address indexed account, uint256 amount);
    event BudgetSet(address indexed user, uint256 budget);
    event GasSponsored(address indexed user, uint256 amount);

    modifier onlyOwner() {
        // TODO etudiant : verifier que msg.sender == owner
        pass;
        _;
    }

    constructor(address _signer) {
        owner = msg.sender;
        verifyingSigner = _signer;
    }

    function deposit() external payable {
        deposits[msg.sender] += msg.value;
        emit Deposited(msg.sender, msg.value);
    }

    function setUserBudget(address user, uint256 budget) external onlyOwner {
        // TODO etudiant : definir le budget de l utilisateur
        pass;
    }

    function getUserBudget(address user) external view returns (uint256) {
        // TODO etudiant : retourner le budget de l utilisateur
        return 0;
    }

    function sponsorGas(address user, uint256 gasAmount) external onlyOwner {
        // TODO etudiant : verifier que le budget de l utilisateur est suffisant
        // Indice : require(spent[user] + gasAmount <= budgets[user], "budget exceeded")
        // TODO etudiant : verifier que le paymaster a assez de fonds
        // TODO etudiant : incrementer spent[user]
        // TODO etudiant : debiter deposits[owner]
        pass;
    }

    function getRemainingBudget(address user) external view returns (uint256) {
        // TODO etudiant : retourner budgets[user] - spent[user]
        return 0;
    }

    receive() external payable {}
}
"""

print("Exercice a completer : BudgetPaymaster avec budget par utilisateur")

Exercice a completer : BudgetPaymaster avec budget par utilisateur


---

## 4. Flux de Transaction ERC-4337

In [6]:
# Flux ERC-4337
print("""
FLUX DE TRANSACTION ERC-4337

1. CREATION
   User -> Crée UserOperation -> Signe avec clé privée

2. SOUMISSION
   UserOp -> Bundler (via RPC ou API)

3. VALIDATION
   Bundler -> simulateValidation() sur EntryPoint
   EntryPoint -> appelle validateUserOp() sur Smart Account
   Smart Account -> vérifie signature

4. EXECUTION
   Bundler -> agrège plusieurs UserOps
   Bundler -> handleOps() sur EntryPoint
   EntryPoint -> execute les opérations validées

AVANTAGES:
- Wallets sans seed phrase (social recovery)
- Gas sponsoring (paymasters)
- Batch transactions
- Transactions sans ETH (pay with tokens)
- Logic programmable (2FA, spending limits)

ADDRESSES EntryPoint:
- Mainnet: 0x5FF137D4b0FDCD49DcA30c7CF57E578a026d2789
- Sepolia: 0x5FF137D4b0FDCD49DcA30c7CF57E578a026d2789
- Goerli: 0x5FF137D4b0FDCD49DcA30c7CF57E578a026d2789
""")


FLUX DE TRANSACTION ERC-4337

1. CREATION
   User -> Crée UserOperation -> Signe avec clé privée

2. SOUMISSION
   UserOp -> Bundler (via RPC ou API)

3. VALIDATION
   Bundler -> simulateValidation() sur EntryPoint
   EntryPoint -> appelle validateUserOp() sur Smart Account
   Smart Account -> vérifie signature

4. EXECUTION
   Bundler -> agrège plusieurs UserOps
   Bundler -> handleOps() sur EntryPoint
   EntryPoint -> execute les opérations validées

AVANTAGES:
- Wallets sans seed phrase (social recovery)
- Gas sponsoring (paymasters)
- Batch transactions
- Transactions sans ETH (pay with tokens)
- Logic programmable (2FA, spending limits)

ADDRESSES EntryPoint:
- Mainnet: 0x5FF137D4b0FDCD49DcA30c7CF57E578a026d2789
- Sepolia: 0x5FF137D4b0FDCD49DcA30c7CF57E578a026d2789
- Goerli: 0x5FF137D4b0FDCD49DcA30c7CF57E578a026d2789



---

## 5. Exemple guide et Exercice

L'exemple guidé ci-dessous montre un contrat résolu (solution étudiante). Un exercice à compléter suit.

In [11]:
# Exemple guide : Social Recovery Account (solution damiendth + godric + kim, TP 2026)
# Version standalone (sans dependances ERC-4337)
# Contrat avec recuperation sociale via des gardiens : startRecovery, approveRecovery, cancelRecovery
# Solution etudiante : Godric Bouteloup, Kim Tayant-Serrat, Damien Duthou (TP 2026)

EXERCISE_RECOVERY = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

/// @title SocialRecoveryAccount
/// @notice Smart account avec recuperation sociale via des gardiens.
/// Si le owner perd sa cle, les gardiens peuvent voter pour changer le owner.
contract SocialRecoveryAccount {
    address public owner;
    address[] public guardians;
    mapping(address => bool) public isGuardian;
    uint256 public guardianCount;
    uint256 public threshold;
    uint256 public nonce;

    // Recovery state
    address public pendingNewOwner;
    uint256 public recoveryApprovals;
    mapping(address => bool) public hasApprovedRecovery;

    event GuardianAdded(address indexed guardian);
    event RecoveryStarted(address indexed newOwner);
    event RecoveryApproved(address indexed guardian);
    event RecoveryCompleted(address indexed newOwner);
    event Executed(address indexed dest, uint256 value);

    modifier onlyOwner() {
        require(msg.sender == owner, "not owner");
        _;
    }

    modifier onlyGuardian() {
        require(isGuardian[msg.sender], "not guardian");
        _;
    }

    constructor(
        address _owner,
        address[] memory _guardians,
        uint256 _threshold
    ) {
        require(_owner != address(0), "zero owner");
        require(_threshold > 0, "threshold zero");
        require(_threshold <= _guardians.length, "threshold too high");
        owner = _owner;
        threshold = _threshold;
        for (uint256 i = 0; i < _guardians.length; i++) {
            address g = _guardians[i];
            require(g != address(0), "zero guardian");
            require(!isGuardian[g], "duplicate guardian");
            isGuardian[g] = true;
            guardians.push(g);
            guardianCount++;
            emit GuardianAdded(g);
        }
    }

    function execute(address dest, uint256 value, bytes calldata data) external onlyOwner {
        nonce++;
        (bool success, ) = dest.call{value: value}(data);
        require(success, "execution failed");
        emit Executed(dest, value);
    }

    function startRecovery(address newOwner) external onlyGuardian {
        require(pendingNewOwner == address(0), "recovery in progress");
        require(newOwner != address(0), "zero new owner");
        pendingNewOwner = newOwner;
        hasApprovedRecovery[msg.sender] = true;
        recoveryApprovals = 1;
        emit RecoveryStarted(newOwner);
        emit RecoveryApproved(msg.sender);
    }

    function approveRecovery() external onlyGuardian {
        require(pendingNewOwner != address(0), "no recovery");
        require(!hasApprovedRecovery[msg.sender], "already approved");
        hasApprovedRecovery[msg.sender] = true;
        recoveryApprovals++;
        emit RecoveryApproved(msg.sender);
        if (recoveryApprovals >= threshold) {
            address newOwner = pendingNewOwner;
            owner = newOwner;
            for (uint256 i = 0; i < guardians.length; i++) {
                hasApprovedRecovery[guardians[i]] = false;
            }
            pendingNewOwner = address(0);
            recoveryApprovals = 0;
            emit RecoveryCompleted(newOwner);
        }
    }

    function cancelRecovery() external onlyOwner {
        require(pendingNewOwner != address(0), "no recovery");
        for (uint256 i = 0; i < guardians.length; i++) {
            hasApprovedRecovery[guardians[i]] = false;
        }
        pendingNewOwner = address(0);
        recoveryApprovals = 0;
    }

    receive() external payable {}
}
'''

guardian1 = w3.eth.accounts[1]
guardian2 = w3.eth.accounts[2]
guardian3 = w3.eth.accounts[3]
new_owner = w3.eth.accounts[4]

recovery, receipt = compile_and_deploy(
    w3, EXERCISE_RECOVERY, [deployer, [guardian1, guardian2, guardian3], 2], deploy_from=deployer
)
print(f"Owner initial: {recovery.functions.owner().call()}")
print(f"Gardiens: {recovery.functions.guardianCount().call()}, threshold: {recovery.functions.threshold().call()}")

tx = w3.eth.send_transaction({'from': deployer, 'to': recovery.address, 'value': w3.to_wei(2, 'ether')})
w3.eth.wait_for_transaction_receipt(tx)

dest = w3.eth.accounts[5]
balance_before = w3.eth.get_balance(dest)
tx = recovery.functions.execute(dest, w3.to_wei(1, 'ether'), b'').transact({'from': deployer})
w3.eth.wait_for_transaction_receipt(tx)
balance_after = w3.eth.get_balance(dest)
print(f"Execute: {w3.from_wei(balance_after - balance_before, 'ether')} ETH transfere, nonce: {recovery.functions.nonce().call()}")

tx = recovery.functions.startRecovery(new_owner).transact({'from': guardian1})
w3.eth.wait_for_transaction_receipt(tx)
print(f"Recovery demarree par guardian1, approvals: {recovery.functions.recoveryApprovals().call()}")

tx = recovery.functions.approveRecovery().transact({'from': guardian2})
w3.eth.wait_for_transaction_receipt(tx)
print(f"Approuvee par guardian2 -> nouveau owner: {recovery.functions.owner().call()}")
print(f"Nouveau owner attendu:                    {new_owner}")
print(f"Etat recovery reset (pendingNewOwner): {recovery.functions.pendingNewOwner().call()}")

Owner initial: 0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266
Gardiens: 3, threshold: 2
Execute: 1 ETH transfere, nonce: 1
Recovery demarree par guardian1, approvals: 1
Approuvee par guardian2 -> nouveau owner: 0x15d34AAf54267DB7D7c367839AAf71A00a2C6A65
Nouveau owner attendu:                    0x15d34AAf54267DB7D7c367839AAf71A00a2C6A65
Etat recovery reset (pendingNewOwner): 0x0000000000000000000000000000000000000000


### Analyse : Social Recovery Account

**Solution validée** (Godric Bouteloup, Kim Tayant-Serrat, Damien Duthou, TP 2026) : Le contrat `SocialRecoveryAccount` implémente un système complet de récupération sociale.

| Fonction | Modificateur | Mécanisme |
|----------|-------------|-----------|
| `constructor(owner, guardians[], threshold)` | - | Initialise owner + gardiens avec vérification doublons |
| `execute(dest, value, data)` | `onlyOwner` | Exécution de transaction avec incrément nonce |
| `startRecovery(newOwner)` | `onlyGuardian` | Premier vote + enregistrement pendingNewOwner |
| `approveRecovery()` | `onlyGuardian` | Vote supplémentaire, si `approvals >= threshold` → transfert ownership |
| `cancelRecovery()` | `onlyOwner` | Reset complet de l'état de récupération |

**Flux de récupération** :
1. Un gardien appelle `startRecovery(newOwner)` → `pendingNewOwner` enregistré, premier vote compté
2. D'autres gardiens appellent `approveRecovery()` → votes accumulés
3. Quand `recoveryApprovals >= threshold` → ownership transféré automatiquement, état reset
4. L'owner peut `cancelRecovery()` à tout moment avant la complétion

**Points clés** :
- Le pattern `threshold` sur N gardiens évite la perte de compte si un gardien est compromis
- Le `require(!isGuardian[g])` dans le constructeur empêche les doublons
- `hasApprovedRecovery` empêche un gardien de voter deux fois
- Le reset complet dans `approveRecovery()` et `cancelRecovery()` nettoie tous les mappings

### Exercice : Daily Spending Limit

Créez un contrat `SpendingLimitAccount` qui hérite de `StandaloneSmartAccount` (section 2) et ajoute une limite de dépense journalière.

In [12]:
# Exercice : Daily Spending Limit
# TODO etudiant : implementer un contrat SpendingLimitAccount
# Indice : inspirez-vous du StandaloneSmartAccount (section 2) pour la structure de base
# Etape 1 : ajouter un dailyLimit (uint256) et spentToday (mapping address => uint256)
# Etape 2 : ajouter lastResetDate (mapping address => uint256 timestamp)
# Etape 3 : override execute() pour verifier spentToday + amount <= dailyLimit
# Etape 4 : si la date a change (block.timestamp / 86400 != lastResetDate), reset spentToday

EXERCISE_SPENDING_LIMIT = """
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract SpendingLimitAccount {
    address public owner;
    uint256 public nonce;
    uint256 public dailyLimit;

    // TODO etudiant : ajouter les variables d'etat pour le tracking quotidien
    // Indice : spentToday, lastResetDay (jour = block.timestamp / 86400)

    event Executed(address indexed dest, uint256 value, bytes data);

    modifier onlyOwner() {
        // TODO etudiant : verifier que msg.sender == owner
        pass;
        _;
    }

    constructor(uint256 _dailyLimit) {
        // TODO etudiant : initialiser owner et dailyLimit
    }

    function execute(address dest, uint256 value, bytes calldata data) external onlyOwner {
        // TODO etudiant : verifier et mettre a jour le plafond quotidien
        // Etape 1 : verifier si le jour a change, si oui reset spentToday
        // Etape 2 : verifier que spentToday + value <= dailyLimit
        // Etape 3 : incrementer nonce, executer l'appel, mettre a jour spentToday
        pass
    }

    function setDailyLimit(uint256 newLimit) external onlyOwner {
        // TODO etudiant : permettre au owner de changer la limite
        pass
    }

    function getSpentToday() public view returns (uint256) {
        // TODO etudiant : retourner la depense du jour
        return 0;
    }

    receive() external payable {}
}
"""

print("Exercice a completer : SpendingLimitAccount avec plafond quotidien")

Exercice a completer : SpendingLimitAccount avec plafond quotidien


---

## 6. Resume

| Concept | Description |
|---------|-------------|
| UserOperation | Structure de transaction ERC-4337 |
| EntryPoint | Singleton qui traite les UserOps |
| Smart Account | Wallet programmable |
| Paymaster | Sponsor de gas fees |
| Bundler | Empaqueteur de transactions |
| Social Recovery | Recuperation via gardiens |

---

**Notebook suivant** : [SC-11-LLM-Assisted](SC-11-LLM-Assisted.ipynb)

## Resume et perspectives

Ce notebook a demystifie l'account abstraction et le standard ERC-4337, une evolution majeure du modele de compte Ethereum. Nous avons etudie la structure `UserOperation` qui remplace les transactions classiques, avec ses champs `sender`, `nonce`, `callData`, `signature` et `paymasterAndData`. Le `StandaloneSmartAccount` deploye sur anvil a illustre les fonctionnalites fondamentales : execution de transactions (`execute`), batch d'operations (`executeBatch`), changement de proprietaire (`changeOwner`) et compteur de nonce.

Le concept de Paymaster a ete demontre avec un sponsor de gas qui depose des fonds et les alloue aux utilisateurs, simulant le mecanisme de transactions sans frais (gasless). L'exercice de Social Recovery a permis d'appliquer ces concepts a un cas pratique critique : la recuperation de compte via un systeme de gardiens avec un seuil de votes requis, eliminant la dependance aux seed phrases.

Le notebook suivant, [SC-11-LLM-Assisted](SC-11-LLM-Assisted.ipynb), change de perspective en explorant comment les modeles de langage (Claude, GPT) peuvent assister le developpement de smart contracts, de la generation de code a l'audit de securite automatise.

---

[<< DAO Governance](SC-9-DAO-Governance.ipynb) | [LLM Assisted >>](SC-11-LLM-Assisted.ipynb)